In [ ]:
library(Seurat)
library(dplyr)
library(data.table)
library(ggplot2)
source("~/Projects/heads/clustering.r")

In [ ]:
data_dir = "/gpfs/gibbs/pi/braun/zy325"
cellranger_dir = file.path(data_dir,"gex_cellranger")

In [ ]:
### Sample-level QC ###
# Based on web summary (cellranger)
# Samples are filtered due to
# 1. Median genes per cell < 500
# 2. Estimated number of cells < 100
# 3. Non-RCC

rm_samples = paste0("SC_RCC_",c("08","17","25","34","63","77",
                                "69","78","79"))

In [ ]:
### Create SeuratObject ###

samples = list.dirs(cellranger_dir,recursive = F,full.names = F)
samples = samples[-which(samples %in% rm_samples)]

mtx_dirs = file.path(cellranger_dir,samples,"outs","filtered_feature_bc_matrix")
names(mtx_dirs) = gsub("_","",samples)

In [ ]:
mtx = Read10X(data.dir = mtx_dirs)

In [ ]:
obj = CreateSeuratObject(counts = mtx)

In [ ]:
saveRDS(obj,file = file.path(data_dir,"processed","scrcc_raw.rds"))

In [ ]:
obj.list = SplitObject(obj,split.by = "orig.ident")

In [ ]:
saveRDS(obj.list,file = file.path(data_dir,"processed","scrcc_raw_list.rds"))

In [ ]:
### Miya ###

obj_path = "/gpfs/gibbs/pi/braun/mh2632/SC_RCC_Individual_RDS/SC_RCC_merged_20230612.rds"

obj = readRDS(obj_path)

In [ ]:
meta = obj@meta.data
meta = meta[,-grep("pANN_|DF.classifications_",colnames(meta))]

meta = meta %>% mutate(
    name=rownames(meta),
    sample_id3=gsub("_PostQC.*$","",name),
    sample_id3=gsub("_new_mt","",sample_id3))

meta$sample_id3[meta$sample_id3 == "SC_07_NORM"] = "SC_07_NK"
meta$orig.ident[meta$sample_id3 == "SC_64_NORM"] = "SC_64_NORM"

meta = meta %>% mutate(
    sample_id1 = gsub("SC_","SC_RCC_",sample_id3),
    sample_id2 = gsub("_","",sample_id1)) %>%
    select(
        orig.ident,nCount_RNA,nFeature_RNA,percent.mt,name,
        sample_id1,sample_id2,sample_id3,DoubletScore,DoubletCall)

In [ ]:
obj@meta.data = meta

In [ ]:
### Batch ###

batch = as.data.frame(fread(file.path(data_dir,"metadata","batch.csv")))

meta = left_join(obj@meta.data,batch,by="sample_id1")
rownames(meta) = rownames(obj@meta.data)

obj@meta.data = meta

In [ ]:
obj = obj[,-which(obj$sample_id1 %in% rm_samples)]

In [ ]:
obj[["RNA"]] = as(obj[["RNA"]],"Assay5")

In [ ]:
obj = clustering(obj,
                plot_QC_metrics = F,
                group.by.vars = "batch_lab",dims = 1:30)

In [ ]:
saveRDS(obj,file=file.path(data_dir,"processed","scrcc_miya_clustered.rds"))

In [ ]:
obj = readRDS(file.path(data_dir,"processed","scrcc_miya_clustered.rds"))

In [ ]:
DimPlot(obj,group.by = "batch_lab") #+ NoLegend()

In [ ]:
DimPlot(obj,group.by = "batch_seq_rna") #+ NoLegend()

In [ ]:
options(repr.plot.width=15,repr.plot.height=10)
VlnPlot(obj,features = c("MZB1","JCHAIN","SDC1","CSF3R","FPR1","FCGR3B","TPSAB1","CPA3","MS4A2","HBB","PPBP","EPCAM","ALDOB","PLVAP","ACTA2","PTPRC"),pt.size = 0,stack = T,flip = T)

In [ ]:
############ scrcc_miya_clustered.rds ############

# Find CD45+ clusters
obj$CD45_data = obj@assays$RNA@layers$data[which(rownames(obj)=="PTPRC"),]

immune_clusters = obj@meta.data %>% group_by(seurat_clusters) %>% 
    summarise(CD45_data_q3 = quantile(CD45_data,probs = .75)) %>% 
    filter(CD45_data_q3>0) %>% .$seurat_clusters

# Check if any CD45low immune cells are missed out
# PC - MZB1, JCHAIN, SDC1
# Mast - TPSAB1, CPA3, MS4A2
# Neutrophil - CSF3R, FPR1, FCGR3B

# Check if any CD45+ clusters are contamination
# High RBC - 20
# High Epi - 48, 52, 53
# High Endo - 33

immune_clusters = setdiff(immune_clusters,c(20,33,48,52,53))

In [ ]:
obj$lineage1 = ifelse(obj$`RNA_snn_res.0.5` %in% immune_clusters,"Immune","NonImmune") 

In [ ]:
obj = subset(obj,lineage1=="Immune")

In [ ]:
DimPlot(obj,group.by = "lineage1") + NoLegend()

In [ ]:
FeaturePlot(obj,features= "PTPRC") #+ NoLegend()

In [ ]:
obj@meta.data %>% head

In [ ]:
obj$`RNA_snn_res.0.5` = NULL
obj$seurat_clusters = NULL

In [ ]:
saveRDS(obj,file=file.path(data_dir,"processed","scrcc_immune.rds"))

In [ ]:
m = FindMarkers(obj,`ident.1` = 33,only.pos = T,logfc.threshold = 1)
m %>% filter(p_val_adj<0.01) %>% arrange(desc(avg_log2FC)) %>% filter(abs(pct.1-pct.2)>.1)

In [ ]:
table(obj$seurat_clusters)